In [1]:
import pandas as pd
from sqlalchemy import create_engine
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load data from database
engine = create_engine('postgresql://localhost/steam_analytics')
df = pd.read_sql('SELECT id, review_text, is_recommended FROM reviews LIMIT 50000', engine)
print('Loaded:', len(df), 'reviews')
df.head()

Loaded: 50000 reviews


,id,review_text,is_recommended
0,9681,"2014 Kill someone with a P90 - ""You're a ♥♥♥♥i...",True
1,9682,2023 Pretty insane that a multi billion dollar...,False
2,9683,"2016 At first, when I started playing this gam...",False
3,9684,"2013 If you aren't 21 buy this, it's basicaly ...",True
4,9685,"2021 Learn use AK, all have do is pull down mo...",True


In [2]:
# Run VADER on all reviews
analyzer = SentimentIntensityAnalyzer()

print('Running VADER...')
df['vader_score'] = df['review_text'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

# Convert VADER score to positive/negative prediction
# VADER compound score: above 0.05 = positive, below -0.05 = negative
df['vader_prediction'] = df['vader_score'].apply(
    lambda x: True if x >= 0.05 else False
)

print('Done!')
print(df[['review_text', 'vader_score', 'vader_prediction', 'is_recommended']].head())

Running VADER...
Done!
                                         review_text  vader_score  \
0  2014 Kill someone with a P90 - "You're a ♥♥♥♥i...       0.9998   
1  2023 Pretty insane that a multi billion dollar...      -0.3818   
2  2016 At first, when I started playing this gam...      -0.2023   
3  2013 If you aren't 21 buy this, it's basicaly ...       0.0000   
4  2021 Learn use AK, all have do is pull down mo...       0.4927   

   vader_prediction  is_recommended  
0              True            True  
1             False           False  
2             False           False  
3             False            True  
4              True            True  


In [4]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

# Calculate VADER accuracy
accuracy = accuracy_score(df['is_recommended'], df['vader_prediction'])
f1 = f1_score(df['is_recommended'], df['vader_prediction'])

print(f'VADER Accuracy: {accuracy*100:.1f}%')
print(f'VADER F1 Score: {f1*100:.1f}%')
print()
print(classification_report(df['is_recommended'], df['vader_prediction']))

VADER Accuracy: 64.3%
VADER F1 Score: 70.6%

              precision    recall  f1-score   support

       False       0.54      0.55      0.54     19577
        True       0.71      0.71      0.71     30423

    accuracy                           0.64     50000
   macro avg       0.63      0.63      0.63     50000
weighted avg       0.64      0.64      0.64     50000



In [5]:
from textblob import TextBlob
from sklearn.metrics import classification_report, f1_score, accuracy_score

print('Running TextBlob...')
df['textblob_score'] = df['review_text'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)

df['textblob_prediction'] = df['textblob_score'].apply(
    lambda x: True if x >= 0.0 else False
)

accuracy_tb = accuracy_score(df['is_recommended'], df['textblob_prediction'])
f1_tb = f1_score(df['is_recommended'], df['textblob_prediction'])

print(f'TextBlob Accuracy: {accuracy_tb*100:.1f}%')
print(f'TextBlob F1 Score: {f1_tb*100:.1f}%')
print()
print(classification_report(df['is_recommended'], df['textblob_prediction']))


Running TextBlob...
TextBlob Accuracy: 68.4%
TextBlob F1 Score: 75.6%

              precision    recall  f1-score   support

       False       0.62      0.50      0.55     19577
        True       0.71      0.80      0.76     30423

    accuracy                           0.68     50000
   macro avg       0.67      0.65      0.65     50000
weighted avg       0.68      0.68      0.68     50000

